# Day 09. Exercise 02
# Metrics

## 0. Imports

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid, cross_val_score
from tqdm.notebook import tqdm
import importlib
import tqdm.notebook as tqdm_notebook
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [3]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
old_df = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = old_df["dayofweek"]
X = df.drop(columns=["dayofweek"])
y = df["dayofweek"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [7]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="auto",
    class_weight=None,
    probability=True,
    random_state=21
)

svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)
y_proba = svm_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred,average="weighted")
roc_auc = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovo",
    average="weighted",
    labels=svm_model.classes_
)

print(f"accuracy is {accuracy:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [9]:
tree_model = DecisionTreeClassifier(
    max_depth=20,
    class_weight="balanced",
    criterion="entropy",
    random_state=21
)

tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
tree_proba = tree_model.predict_proba(X_test)

tree_accuracy = accuracy_score(y_test, tree_pred)
tree_precision = precision_score(y_test, tree_pred, average="weighted")
tree_recall = recall_score(y_test, tree_pred, average="weighted")
tree_roc_auc = roc_auc_score(
    y_test,
    tree_proba,
    multi_class="ovo",
    average="weighted",
    labels=tree_model.classes_
)

print(f"accuracy is {tree_accuracy:.5f}")
print(f"precision is {tree_precision:.5f}")
print(f"recall is {tree_recall:.5f}")
print(f"roc_auc is {tree_roc_auc:.5f}")

accuracy is 0.89941
precision is 0.90098
recall is 0.89941
roc_auc is 0.93954


## 4. Random forest

1. The same task for random forest.

In [10]:
forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=32,
    class_weight=None,
    criterion="entropy",
    random_state=21
)

forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
forest_proba = forest_model.predict_proba(X_test)

forest_accuracy = accuracy_score(y_test, forest_pred)
forest_precision = precision_score(y_test, forest_pred, average="weighted")
forest_recall = recall_score(y_test, forest_pred, average="weighted")
forest_roc_auc = roc_auc_score(
    y_test,
    forest_proba,
    multi_class="ovo",
    average="weighted",
    labels=forest_model.classes_
)

print(f"accuracy is {forest_accuracy:.5f}")
print(f"precision is {forest_precision:.5f}")
print(f"recall is {forest_recall:.5f}")
print(f"roc_auc is {forest_roc_auc:.5f}")

accuracy is 0.93195
precision is 0.93387
recall is 0.93195
roc_auc is 0.98975


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [27]:
result_df = pd.DataFrame({
    "true_day": y_test,
    "predict_day": forest_pred
})
result_df["error"] = result_df["true_day"] != result_df["predict_day"]
errors_by_day = result_df.groupby("true_day")["error"].sum()
total_by_day = df["dayofweek"].value_counts()

error_percent = (errors_by_day / total_by_day * 100).sort_values(ascending=False)
error_percent


true_day
0    5.147059
4    2.884615
5    1.476015
1    1.459854
2    1.342282
3    0.505051
6    0.280899
dtype: float64

Количество ошибок наибольшее для понедельника

In [36]:
# для пользователей
user_columns = df.filter(like="uid_").columns
error_indexes = result_df.index[result_df["error"]]
errors_by_user = df.loc[error_indexes, user_columns].sum()
total_by_user = df[user_columns].sum()
user_error_percent = (
    errors_by_user / total_by_user * 100
).sort_values(ascending=False)
print(user_error_percent)

uid_user_6     8.333333
uid_user_16    6.250000
uid_user_27    4.347826
uid_user_19    3.296703
uid_user_18    2.857143
uid_user_3     2.816901
uid_user_31    2.666667
uid_user_30    2.564103
uid_user_24    1.785714
uid_user_25    1.666667
uid_user_2     1.652893
uid_user_29    1.562500
uid_user_10    1.408451
uid_user_4     1.063830
uid_user_14    0.757576
uid_user_26    0.000000
uid_user_28    0.000000
uid_user_7     0.000000
uid_user_0     0.000000
uid_user_22    0.000000
uid_user_23    0.000000
uid_user_1     0.000000
uid_user_21    0.000000
uid_user_20    0.000000
uid_user_17    0.000000
uid_user_15    0.000000
uid_user_13    0.000000
uid_user_12    0.000000
uid_user_11    0.000000
uid_user_8     0.000000
dtype: float64


Больше всего ошибок для пользователя с id 6

In [66]:
lab_df = df.copy().filter(like="labname_")
lab_df["lab"] = lab_df.idxmax(axis=1)
error_sum = lab_df.loc[error_indexes].groupby("lab").sum().max()
sum_lab = lab_df.groupby("lab").sum().max()
lab_error_percent = error_sum / sum_lab * 100
print(lab_error_percent)


labname_code_rvw      1.219512
labname_lab02         0.000000
labname_lab03       100.000000
labname_lab03s      100.000000
labname_lab05s        2.777778
labname_laba04        3.370787
labname_laba04s       0.961538
labname_laba05        0.450450
labname_laba06        2.083333
labname_laba06s       1.639344
labname_project1      0.946372
dtype: float64


lab3 и lab03s

In [67]:
best_model = forest_model
joblib.dump(best_model, "../data/metrics_random_forest.joblib")

['../data/metrics_random_forest.joblib']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [67]:
def evaluate_models(
    models, params_list, X_train, y_train, X_test, y_test
):
    results = {}

    for model_class, params in zip(models, params_list):
        model = model_class(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall = recall_score(y_test, y_pred, average="weighted")
        roc_auc = roc_auc_score(
            y_test,
            y_proba,
            multi_class="ovo",
            average="weighted",
            labels=model.classes_
        )

        results[model_class.__name__] = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "roc_auc": roc_auc
        }

        print(model_class.__name__)
        print(f"accuracy is {accuracy:.5f}")
        print(f"precision is {precision:.5f}")
        print(f"recall is {recall:.5f}")
        print(f"roc_auc is {roc_auc:.5f}")

    return results

In [68]:
models = [SVC, DecisionTreeClassifier, RandomForestClassifier]

params_list = [
    {
        "kernel": "rbf",
        "C": 10,
        "gamma": "auto",
        "class_weight": None,
        "probability": True,
        "random_state": 21
    },
    {
        "max_depth": 20,
        "class_weight": "balanced",
        "criterion": "entropy",
        "random_state": 21
    },
    {
        "n_estimators": 100,
        "max_depth": 32,
        "class_weight": None,
        "criterion": "entropy",
        "random_state": 21
    }
]

model_results = evaluate_models(
    models, params_list, X_train, y_train, X_test, y_test
)

SVC
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
DecisionTreeClassifier
accuracy is 0.89941
precision is 0.90098
recall is 0.89941
roc_auc is 0.93954
RandomForestClassifier
accuracy is 0.93195
precision is 0.93387
recall is 0.93195
roc_auc is 0.98975


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
